In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import datetime
import os
import csv
import threading


LOG_FILE = 'log.csv'
MAX_RECORDS = 100
LOCK = threading.Lock()


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


server = SimpleXMLRPCServer(("localhost", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True)


def _count_records():
    if not os.path.exists(LOG_FILE):
        return 0
    with open(LOG_FILE, encoding='utf-8') as f:
        return sum(1 for _ in f)


def _rotate_if_needed():
    if _count_records() <= MAX_RECORDS:
        return
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    new_name = f"log_{ts}.csv"
    os.rename(LOG_FILE, new_name)
    print(f"[stats] Ротация журнала: {LOG_FILE} -> {new_name}")


def register_event(event_type, details=""):
    with LOCK:
        _rotate_if_needed()
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(
                f"{event_type},{details},"
                f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
            )
    return True


def get_events(event_type="", dt_from="", dt_to=""):
    if not os.path.exists(LOG_FILE):
        return []

    rows = []
    with open(LOG_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            parts = line.split(',', 2)
            etype, details, ts = parts

            if event_type and etype != event_type:
                continue
            if dt_from and ts < dt_from:
                continue
            if dt_to and ts > dt_to:
                continue

            rows.append({
                "event_type": etype,
                "details": details,
                "timestamp": ts,
            })
    return rows


def get_event_types():
    if not os.path.exists(LOG_FILE):
        return []
    types = set()
    with open(LOG_FILE, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split(',', 2)
            if parts and parts[0]:
                types.add(parts[0])
    return sorted(types)


def get_log_size():
    return _count_records()


server.register_function(register_event, 'register_event')
server.register_function(get_events,     'get_events')
server.register_function(get_event_types,'get_event_types')
server.register_function(get_log_size,   'get_log_size')


print("Listening on port 8018...")
server.serve_forever()

Listening on port 8018...


127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:25] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:38] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 19:21:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/202

KeyboardInterrupt: 